In [10]:
# LLM Zoomcamp - Homework 2
import numpy as np
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

# Initialize embedder and check Q1
embedder = Embedder()
query = 'How does approximate nearest neighbor search work?'
v = embedder.encode(query)
print(f'Q1 Value (v[0]): {v[0]:.2f}')

# Download documents and check Q2
reader = GithubRepositoryDataReader(repo_owner="DataTalksClub", repo_name="llm-zoomcamp", commit_id="8c1834d", allowed_extensions={"md"}, filename_filter=lambda path: "/lessons/" in path)
documents = [file.parse() for file in reader.read()]
target_doc = next(doc for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")
v_doc = embedder.encode(target_doc["content"])
print(f'Q2 Cosine Similarity: {np.dot(v, v_doc):.2f}')

# Chunk documents and check Q3
chunks = chunk_documents(documents, size=2000, step=1000)
contents = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(contents)
scores = X.dot(v)
print(f'Q3 Highest-scoring file: {chunks[np.argmax(scores)]["filename"]}')

# Setup VectorSearch and check Q4
v_search = VectorSearch()
v_search.fit(X, chunks)
v_q4 = embedder.encode('What metric do we use to evaluate a search engine?')
print('Q4 First Result:', v_search.search(v_q4, num_results=1)[0]["filename"])

# Setup Keyword Index and check Q5
text_index = Index(text_fields=["content"])
text_index.fit(chunks)
query_q5 = 'How do I store vectors in PostgreSQL?'
v_files = {doc["filename"] for doc in v_search.search(embedder.encode(query_q5), num_results=5)}
t_files = {doc["filename"] for doc in text_index.search(query_q5, num_results=5)}
print('Q5 Unique to vector search:', v_files - t_files)

# Define RRF and check Q6
def rrf(lists, k=60):
    scores, docs = {}, {}
    for l in lists:
        for rank, doc in enumerate(l):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    return [docs[k] for k in sorted(scores, key=scores.get, reverse=True)]

q6_v = v_search.search(embedder.encode('How do I give the model access to tools?'), num_results=10)
q6_t = text_index.search('How do I give the model access to tools?', num_results=10)
print('Q6 Top Hybrid Ranked File:', rrf([q6_v, q6_t])[0]["filename"])

Q1 Value (v[0]): -0.02
Q2 Cosine Similarity: 0.36
Q3 Highest-scoring file: 02-vector-search/lessons/07-sqlitesearch-vector.md
Q4 First Result: 04-evaluation/lessons/05-search-metrics.md
Q5 Unique to vector search: {'02-vector-search/lessons/08-pgvector.md'}
Q6 Top Hybrid Ranked File: 01-agentic-rag/lessons/13-function-calling.md
